# Calibrated URDFs for UR arms 🎯

Every physical UR arm has its own *calibrated* DH parameters (stored on the
controller), which differ slightly from the manufacturer's nominal DH parameters
used by the `airo_models` URDFs (loaded via `add_manipulator`). The gap is small
(order 1-2 mm at the TCP) but matters for precision tasks.

This notebook builds a calibrated URDF for **`tool0` IK accuracy only**. It has no
visual or collision geometry, on purpose (see section 6) -- for collision checking
and visualization, always use the regular `airo_models` mesh model instead.

## Why not just use analytic IK?

`ur-analytic-ik` (used elsewhere in `airo-drake`/`airo-planner`) has the *nominal*
DH baked into its closed-form solution at compile time, so it can't be pointed at
a specific robot's calibrated parameters. It's still the right tool for picking
*which* joint-configuration branch to use (fast, deterministic, reliable near
singularities/joint limits) -- numerical IK alone is not.

The intended pattern is therefore a **two-stage IK**:

1. **Seed**: analytic IK (nominal DH) picks the branch for the target pose.
2. **Refine**: Drake's numerical `InverseKinematics`, run on a *calibrated* URDF
   (built in this notebook), seeded at the analytic solution with a
   stay-near-seed cost, converges to the accurate joints without flipping branches.

Both stages are provided by `airo_drake.CalibratedKinematics`: its
`inverse_kinematics_closest` does the refine alone, or the full two-stage
analytic-seed-then-refine when constructed with `analytic_ik_model`. Section 5
demonstrates it; to run the full analytic → refine → control-box comparison on a
real arm, see `scripts/manual_calibrated_ik_hardware_test.py`.

## 1. Get the robot's calibrated DH parameters

`airo_drake.read_calibrated_dh` connects to the UR control box's primary interface
(TCP port 30001, no ROS needed) and reads back the calibrated DH. The robot only
needs to be powered on.

If no robot is reachable, this cell falls back to the *nominal* UR5e DH so the rest
of the notebook still runs -- but note that a URDF built from nominal DH carries
**no calibration** (see the `warn_if_uncalibrated` check below).

In [ ]:
import numpy as np
from airo_drake import read_calibrated_dh

ROBOT_IP = None  # e.g. "10.42.0.162" -- set this to try a real robot

if ROBOT_IP is not None:
    dh = read_calibrated_dh(ROBOT_IP)
else:
    pi = np.pi
    dh = {
        "theta": [0] * 6,
        "a": [0, -0.425, -0.3922, 0, 0, 0],
        "d": [0.1625, 0, 0, 0.1333, 0.0997, 0.0996],
        "alpha": [pi / 2, 0, 0, pi / 2, -pi / 2, 0],
        "calibration_status": 0,  # nominal, not read from a robot
    }

dh

### Save the DH for offline reuse

Snapshot the raw DH so you can version it and rebuild the URDF later without the
robot attached.

In [ ]:
from airo_drake import save_calibrated_dh, load_calibrated_dh, warn_if_uncalibrated

save_calibrated_dh(dh, "ur5e_dh.json")

dh_reloaded, meta = load_calibrated_dh("ur5e_dh.json")
warn_if_uncalibrated(meta)  # only prints for the nested robot_calibration.json format
dh_reloaded

## 2. Convert the DH parameters to a URDF

`calibrated_dh_to_urdf` writes a flat URDF whose forward kinematics match the DH
model to ~1e-9 (see `test/test_ur_calibration.py`). It has no meshes and no visual or
collision geometry at all: `base_link` is DH frame 0 and `tool0` is DH frame 6,
matching `getActualTCPPose()` / `ur-analytic-ik`'s frames directly. This URDF is
for `tool0` IK only -- see section 5.

In [ ]:
from airo_drake import calibrated_dh_to_urdf

urdf_path = "ur5e_calibrated.urdf"
with open(urdf_path, "w") as f:
    f.write(calibrated_dh_to_urdf(dh, robot_name="ur5e"))

print(open(urdf_path).read()[:500], "...")

## 3. Build a `CalibratedKinematics` -- weld with **identity**

Unlike the ROS-normalized `airo_models` UR URDFs, this URDF's link frames are the
raw DH frames, so `base_link` must be welded to the world with an **identity**
transform -- *not* `airo_drake.X_URBASE_ROSBASE` (the 180° weld `add_manipulator`
applies for `airo_models` URDFs). `airo_drake.CalibratedKinematics` does exactly this
internally when built from a DH dict, so we construct it directly instead of
building the Drake scene by hand.

In [ ]:
from airo_drake import CalibratedKinematics

calibrated_kinematics = CalibratedKinematics(dh, robot_name="ur5e")

### Sanity check: Drake FK matches the DH FK

In [ ]:
from airo_drake.ur_calibration.conversion import fk_dh

q = np.zeros(6)
X_W_tool0 = calibrated_kinematics.forward_kinematics(q)

X_dh = fk_dh(dh, q)
error = np.abs(X_W_tool0 - X_dh).max()
print(f"max abs FK error: {error:.2e}")
assert error < 1e-6

## 4. Compare `tool0` with the nominal `airo_models` model

A purely numeric check (no visualization needed): build the regular, mesh-based
`airo_models` UR5e -- welded with `airo_drake.X_URBASE_ROSBASE`, unlike this
notebook's calibrated model, which uses identity -- and compare its `tool0` pose
to the calibrated model's, at the same joint configuration. With the nominal-DH
fallback used above, they should match almost exactly; with a real robot's
calibrated DH, expect the ~1-2 mm difference this whole notebook exists to close.

In [ ]:
import airo_models
from pydrake.planning import RobotDiagramBuilder

from airo_drake import X_URBASE_ROSBASE

nominal_builder = RobotDiagramBuilder()
nominal_plant = nominal_builder.plant()
nominal_parser = nominal_builder.parser()

nominal_index = nominal_parser.AddModels(airo_models.get_urdf_path("ur5e"))[0]
nominal_plant.WeldFrames(
    nominal_plant.world_frame(), nominal_plant.GetFrameByName("base_link", nominal_index), X_URBASE_ROSBASE
)

nominal_diagram = nominal_builder.Build()
nominal_context = nominal_diagram.CreateDefaultContext()
del nominal_builder

q_pose = np.deg2rad([0, -45, -90, -90, 90, 0])

nominal_plant_context = nominal_plant.GetMyContextFromRoot(nominal_context)
nominal_plant.SetPositions(nominal_plant_context, nominal_index, q_pose)
X_W_tool0_nominal = nominal_plant.GetFrameByName("tool0", nominal_index).CalcPoseInWorld(nominal_plant_context)

X_W_tool0_calibrated = calibrated_kinematics.forward_kinematics(q_pose)

print("nominal tool0:    ", X_W_tool0_nominal.translation())
print("calibrated tool0: ", X_W_tool0_calibrated[:3, 3])
print("difference (m):   ", np.linalg.norm(X_W_tool0_nominal.translation() - X_W_tool0_calibrated[:3, 3]))

## 5. Two-stage IK: analytic seed → calibrated refine

`CalibratedKinematics`, constructed with `analytic_ik_model`, runs analytic IK (nominal
DH, picks the branch) and then refines it numerically on the calibrated model, all
through the same `inverse_kinematics_closest` call. To see the effect **without a
robot**, we emulate a calibration by perturbing the nominal DH slightly and treat
that model's forward kinematics as "reality". With a real robot's DH (set
`ROBOT_IP` in section 1) the same code shows the real ~1-2 mm gap closing.

In [ ]:
import ur_analytic_ik

from airo_drake import CalibratedKinematics

# Emulate a calibration so the effect is visible without a robot (skip this with real DH).
rng = np.random.default_rng(0)
emulated_dh = {
    key: (np.asarray(dh[key], dtype=float) + rng.normal(0, 1e-3, 6)).tolist()
    for key in ("theta", "a", "d", "alpha")
}

# CalibratedKinematics from the emulated DH, with the analytic model wired in for the two-stage seed.
emulated_kinematics = CalibratedKinematics(emulated_dh, "ur5e", analytic_ik_model=ur_analytic_ik.ur5e)

# A reachable target = the emulated-calibrated robot's true pose at some configuration.
q_true = np.deg2rad([10, -80, 90, -100, -90, 20])
X_target = emulated_kinematics.forward_kinematics(q_true)
q_seed = q_true + np.deg2rad(3)  # a nearby seed, e.g. the robot's current configuration

# Stage 1 only (analytic IK, nominal DH) vs the full two-stage refine.
q_analytic = np.asarray(ur_analytic_ik.ur5e.inverse_kinematics_closest(X_target, *q_seed)[0], dtype=float).reshape(-1)
result = emulated_kinematics.inverse_kinematics_closest(X_target, q_seed=q_seed)
# `result` is a KinematicsResult: the solution plus a flag for whether it stayed on the seed's branch.
q_refined = result.joint_configuration


def error_mm(q):
    return 1000.0 * np.linalg.norm(emulated_kinematics.forward_kinematics(q)[:3, 3] - X_target[:3, 3])


print(f"analytic IK (nominal DH) error: {error_mm(q_analytic):.3f} mm")
print(f"two-stage calibrated IK error:  {error_mm(q_refined):.4f} mm")
print(f"stayed on the seed's branch: {result.is_close_to_seed} (max {np.rad2deg(result.max_seed_deviation):.2f} deg)")

## 6. Which model for what

- **IK on `tool0`** → this calibrated model.
- **Collision checking / visualization** → always the regular `airo_models` mesh
  model (`airo_drake.add_manipulator("ur5e", ...)`). The calibration delta
  (~1-2 mm) is far below normal collision padding, so it stays valid for the real
  robot, and you get the real meshes for free.

This calibrated URDF has no visual or collision geometry, on purpose: real UR
calibration data can contain huge cancelling DH offsets that leave `tool0` exact
while throwing the intermediate link frames far from the physical arm, which would
make any geometry built on them meaningless. Rather than risk a collision checker
or renderer silently trusting bad geometry, this model simply can't be used that
way. See `airo_drake/ur_calibration/conversion.py`'s module docstring for the full
explanation.

The `airo_models` meshes also live in the ROS-normalized link frames, which differ
from this URDF's DH frames, so they wouldn't line up on these links even if we did
attach geometry.